# U-Net Binary Segmentation for Schematic Wire Detection
Train a U-Net to separate handdrawn ink from background.
This is replicated apporach from https://arxiv.org/html/2402.11093v1.

This is to be run on google colab only.

This guide to Segmentation was heavily followed:https://medium.com/@heyamit10/pytorch-segmentation-models-a-practical-guide-5bf973a32e30

## 1. Mount Google Drive


In [2]:
import kagglehub
from pathlib import Path

download_path = kagglehub.dataset_download("johannesbayer/cghd1152")

DATASET_ROOT = Path(download_path)


Using Colab cache for faster access to the 'cghd1152' dataset.


## 2. Install & Import Dependencies

In [3]:
!pip install -q segmentation-models-pytorch albumentations

import os, glob, random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
TEST_SET = 0.1
VAL_SET = 0.10
TRAIN_SET = 0.80

Using device: cuda


## 3. Dataset
Pairs each raw image with its corresponding segmentation mask image.

In [5]:

root_dir = Path(DATASET_ROOT)

seg_images = list(root_dir.glob(f'**/segmentation/*'))

image_pairs = []
missing_count = 0

for seg_path in seg_images:
    # assumes images are "../images/*" relative to the segmentation
    file_stem = seg_path.stem
    images_dir = seg_path.parent.parent / 'images'
    matches = list(images_dir.glob(f"{file_stem}.*"))    
    if matches:
        image_pairs.append((str(matches[0]), str(seg_path)))
           
    else:
        missing_count += 1

print(f"Total segmentation images found: {len(seg_images)}")
print(f"Successfully matched pairs:      {len(image_pairs)}")

if missing_count > 0:
    print(f"Warning: {missing_count} segmentation were missing their raw images file")

Total segmentation images found: 346
Successfully matched pairs:      346


Reference:
The image transformation were created via heavy suggestions from Gemini, the prompt was:What are some good image augmentation combinations to do for training a unet model for making binary segmentation of an image?

In [10]:
PATCH_SIZE = 256
PATCHES_PER_IMAGE = 200  
train_tf = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.ColorJitter(p=0.3),
    A.GaussNoise(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class SchematicSegDataset(Dataset):
    def __init__(self, pairs, transform, patches_per_image=PATCHES_PER_IMAGE, patch_size=PATCH_SIZE):
        self.pairs = pairs
        self.transform = transform
        self.patches_per_image = patches_per_image
        self.patch_size = patch_size
        self.cache = []
        for img_path, seg_path in pairs:
            img = cv2.imread(img_path)
            if img is None:
                raise FileNotFoundError(f'Could not read image: {img_path}')
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise FileNotFoundError(f'Could not read mask: {seg_path}')
            mask = (mask < 127).astype(np.uint8)
            self.cache.append((img,mask))
        print(f"Total Images Loaded: {len(self.cache)}")

    def __len__(self):
        return len(self.cache) * self.patches_per_image

    def __getitem__(self, idx):
        img, mask = self.cache[idx // self.patches_per_image]        
        h, w = img.shape[:2]
        p = self.patch_size
        y = random.randint(0, max(h - p, 0))
        x = random.randint(0, max(w - p, 0))
        img_patch  = img[y:y+p, x:x+p]
        mask_patch = mask[y:y+p, x:x+p]

        img_patch  = cv2.resize(img_patch,  (p, p))
        mask_patch = cv2.resize(mask_patch, (p, p), interpolation=cv2.INTER_NEAREST)

        augmented = self.transform(image=img_patch, mask=mask_patch)
        return augmented['image'], augmented['mask'].long()


random.shuffle(image_pairs)
train_idx = int(0.80 * len(image_pairs))
val_idx = int(0.90 * len(image_pairs)) 

train_pairs = image_pairs[:train_idx]          
val_pairs   = image_pairs[train_idx:val_idx]   
test_pairs  = image_pairs[val_idx:]            

train_ds = SchematicSegDataset(train_pairs, train_tf)
val_ds   = SchematicSegDataset(val_pairs,   val_tf)
test_ds = SchematicSegDataset(val_pairs,   val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=8, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=8, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_ds,   batch_size=32, shuffle=False, num_workers=8, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
print(f'Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}, Test pairs: {len(test_pairs)}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')



Total Images Loaded: 276
Total Images Loaded: 35
Total Images Loaded: 35
Train pairs: 276, Val pairs: 35, Test pairs: 35
Train batches: 1725, Val batches: 219, Test batches: 219


## 4. Model
Using `segmentation_models_pytorch` which gives us a U-Net with a pretrained ResNet34 encoder — stronger than training from scratch and faster to converge.

In [9]:

model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)


loss_fn   = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)



## 5. Training Loop



In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()


def pixel_accuracy(preds, masks):
    preds_bin = (torch.sigmoid(preds.float()) > 0.5).long()
    return (preds_bin.squeeze(1) == masks).float().mean().item()


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_acc = 0, 0
    with torch.set_grad_enabled(train):
        for imgs, masks in loader:
            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE):
                preds = model(imgs)
                loss  = loss_fn(preds, masks.unsqueeze(1).float())
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item()
            total_acc  += pixel_accuracy(preds, masks)
    return total_loss / len(loader), total_acc / len(loader)


EPOCHS = 20
best_val_acc = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {vl_loss:.4f} acc {vl_acc:.4f}')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/unet_old.pth')
        print(f'  -> Saved best model (val acc {best_val_acc:.4f})')


/tmp/ipykernel_44486/2079226392.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_44486/2079226392.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 01/20 | train loss 0.2670 acc 0.9801 | val loss 0.1238 acc 0.9897
  -> Saved best model (val acc 0.9897)
Epoch 02/20 | train loss 0.1392 acc 0.9883 | val loss 0.1118 acc 0.9900
  -> Saved best model (val acc 0.9900)
Epoch 03/20 | train loss 0.1223 acc 0.9895 | val loss 0.1135 acc 0.9895
Epoch 04/20 | train loss 0.1150 acc 0.9900 | val loss 0.1060 acc 0.9902
  -> Saved best model (val acc 0.9902)
Epoch 05/20 | train loss 0.1103 acc 0.9905 | val loss 0.1117 acc 0.9902
Epoch 06/20 | train loss 0.1049 acc 0.9908 | val loss 0.1052 acc 0.9902
Epoch 07/20 | train loss 0.1014 acc 0.9912 | val loss 0.1076 acc 0.9899
Epoch 08/20 | train loss 0.0980 acc 0.9914 | val loss 0.1074 acc 0.9900
Epoch 09/20 | train loss 0.0951 acc 0.9917 | val loss 0.0981 acc 0.9905
  -> Saved best model (val acc 0.9905)
Epoch 10/20 | train loss 0.0933 acc 0.9919 | val loss 0.1039 acc 0.9901


: 

## 6. Test model Performance

## 7. Test Inference
Use the model to infer just a single sample 

In [ ]:
infer_tf = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


def get_wire_mask_unet(image_path, model, threshold=0.5, infer_batch_size=32):
    """Returns a binary uint8 mask (255=ink, 0=background) for the full image."""
    model.eval()
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f'Could not read: {image_path}')
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    patch  = 256
    stride = 224
    accum = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.float32)

    ys = list(range(0, h - patch, stride)) + [max(h - patch, 0)]
    xs = list(range(0, w - patch, stride)) + [max(w - patch, 0)]

    coords  = [(y, x) for y in ys for x in xs]
    crops   = [cv2.resize(img_rgb[y:y+patch, x:x+patch], (patch, patch)) for y, x in coords]
    tensors = torch.stack([infer_tf(image=c)['image'] for c in crops])  # (N,3,256,256)

    preds_all = []
    with torch.no_grad():
        for i in range(0, len(tensors), infer_batch_size):
            batch = tensors[i:i+infer_batch_size].to(DEVICE)
            with torch.autocast(device_type="cuda"):
                preds_all.append(torch.sigmoid(model(batch)).squeeze(1).float().cpu())
    preds_all = torch.cat(preds_all).numpy()  # (N,256,256)

    for (y, x), pred in zip(coords, preds_all):
        pred_r = cv2.resize(pred, (patch, patch))
        accum[y:y+patch, x:x+patch] += pred_r
        count[y:y+patch, x:x+patch] += 1

    prob_map = accum / np.maximum(count, 1)
    mask = ((prob_map > threshold) * 255).astype(np.uint8)
    return mask


# --- Test on a sample image ---
model.load_state_dict(torch.load('/content/drive/MyDrive/unet_best.pth', map_location=DEVICE))

sample_img, sample_seg = val_pairs[2]
mask = get_wire_mask_unet(sample_img, model)
gt   = (cv2.imread(sample_seg, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8) * 255

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(cv2.imread(sample_img), cv2.COLOR_BGR2RGB))
axes[0].set_title('Input Image')
axes[1].imshow(gt, cmap='gray')
axes[1].set_title('Ground Truth Mask')
axes[2].imshow(mask, cmap='gray')
axes[2].set_title('U-Net Predicted Mask')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()
